# 03 -- Collections: 50 Songs About a Keyword

Returns **50 globally most-played songs** matching a given keyword (love, war, happiness, loneliness, money),
using lyrics-based content filtering.

### Algorithm

Three approaches are compared:

1. **Baseline** -- exact keyword match: count occurrences of the keyword in each
   track's lyrics, apply a threshold, then sort by total play count.
2. **Word2Vec** -- expand the keyword with semantically similar tokens via a
   pre-trained word2vec model, then combine their lyric counts.
3. **Classification** -- label tracks as "about X" vs "not about X" using
   keyword presence, train a classifier on the labelled set, then predict
   scores for the remaining tracks.

### Output columns
| Column | Description |
|--------|-------------|
| `rank` | 1-50 index, by total play count (descending) |
| `artist` | Artist name |
| `title` | Track title |
| `play_count` | Total plays across all users |

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.data import MySpotifyRecommender
from src.models.collections import (
    collection_baseline,
    collection_classification_compare,
    collection_word2vec,
)

In [2]:
rs = MySpotifyRecommender.from_files(Path.cwd().parent / "data", download=True, triplets_sample_rows=100_000)

  tracks      (1000000, 4)
  genres      (280831, 3)
  triplets    (100000, 3)
  lyrics_long (16845943, 3)


---
## Research

In [3]:
KEYWORDS = ["love", "war", "happiness", "loneliness", "money"]

### Approach 1 -- Baseline (exact keyword match)

In [4]:
baseline_results = collection_baseline(rs, KEYWORDS, n=1, top_n=50)
for i in range(len(KEYWORDS)):
    keyword = KEYWORDS[i]
    result = baseline_results.get(keyword, "No results found")
    print(f"Keyword: {keyword}")
    display(result)
    print("\n")

Keyword: love


index,artist,title,play_count
u32,str,str,i64
1,"""Dwight Yoakam""","""You're The One""",1689
2,"""Train""","""Marry Me""",488
3,"""Tub Ring""","""Invalid""",415
4,"""Train""","""Hey_ Soul Sister""",403
5,"""Sam Cooke""","""Ain't Misbehavin""",379
…,…,…,…
46,"""U.S. Bombs""","""Cirenda""",88
47,"""Lynyrd Skynyrd""","""Sweet home Alabama""",84
48,"""Florence + The Machine""","""I'm Not Calling You A Liar""",84




Keyword: war


index,artist,title,play_count
u32,str,str,i64
1,"""Sheena Easton""","""Strut (1993 Digital Remaster)""",191
2,"""Eminem / Nate Dogg""","""'Till I Collapse""",121
3,"""Aesop Rock""","""None Shall Pass (Main)""",72
4,"""Creedence Clearwater Revival""","""Fortunate Son""",68
5,"""O.G.C.""","""Gunn Clapp""",68
…,…,…,…
46,"""Kings Of Convenience""","""Renegade""",17
47,"""Promoe""","""Government Music""",17
48,"""Rammstein""","""AMERIKA""",17




Keyword: happiness


index,artist,title,play_count
u32,str,str,i64
1,"""Train""","""Marry Me""",488
2,"""Sam Cooke""","""Ain't Misbehavin""",379
3,"""Counting Crows""","""Mr. Jones""",163
4,"""Radiohead""","""Creep (Explicit)""",160
5,"""Britt Nicole""","""You""",91
…,…,…,…
46,"""The Temper Trap""","""Fools""",22
47,"""Ryan Adams""","""When The Stars Go Blue""",22
48,"""Stars""","""Theme From The Stars""",21




Keyword: loneliness


index,artist,title,play_count
u32,str,str,i64
1,"""Gov't Mule""","""Fool's Moon""",69
2,"""The Black Keys""","""Too Afraid To Love""",40
3,"""Radney Foster""","""The Kindness Of Strangers""",33
4,"""Frightened Rabbit""","""The Loneliness And The Scream""",32
5,"""Paolo Nutini""","""Loving You [Album Version]""",30
…,…,…,…
46,"""Jay Brannan""","""Beautifully""",4
47,"""Gorillaz""","""Slow Country""",4
48,"""I Am Kloot""","""3 Feet Tall (Album Version)""",4




Keyword: money


index,artist,title,play_count
u32,str,str,i64
1,"""Kix""","""Girl Money""",233
2,"""Beastie Boys""","""Unite (2009 Digital Remaster)""",202
3,"""The Verve""","""Bitter Sweet Symphony""",182
4,"""Eminem""","""Mockingbird""",176
5,"""Future Of The Left""","""Yin / Post-Yin""",141
…,…,…,…
46,"""The Cat Empire""","""Hello""",23
47,"""New Order""","""True Faith""",23
48,"""Fabolous""","""Breathe (Amended Album Version…",23


### Approach 2 -- Word2Vec (expanded keywords)

In [5]:
w2v_results = collection_word2vec(rs, KEYWORDS, n=50, top_n=50)
for kw, df in w2v_results.items():
    print(f"\n{'='*50}")
    print(f"  Collection: {kw.upper()}")
    print(f"{'='*50}")
    display(df)


  Collection: LOVE


index,artist,title,play_count
u32,str,str,i64
1,"""Dwight Yoakam""","""You're The One""",1689
2,"""Train""","""Hey_ Soul Sister""",403
3,"""Sam Cooke""","""Ain't Misbehavin""",379
4,"""Lil Wayne / Eminem""","""Drop The World""",363
5,"""Bill Withers""","""Make Love To Your Mind""",322
…,…,…,…
46,"""Lady GaGa""","""Teeth""",72
47,"""Mayday Parade""","""You Be The Anchor That Keeps M…",71
48,"""Alicia Keys""","""Un-thinkable (I'm Ready)""",69



  Collection: WAR


index,artist,title,play_count
u32,str,str,i64
1,"""Culture Club""","""The War Song (2003 Digital Rem…",0
2,"""Frankie Goes To Hollywood""","""War""",0
3,"""Hot Boys""","""Introduction (Hot Boyz/Let Em …",0
4,"""Tiziano Ferro""","""Soul-dier""",0
5,"""Cheryl Cole""","""Fight For This Love""",0



  Collection: HAPPINESS


index,artist,title,play_count
u32,str,str,i64
1,"""Zapp & Roger""","""Computer Love""",10
2,"""Wilson Pickett""","""I'm In Love (Single/LP Version…",9
3,"""Jill Scott""","""It's Love""",8
4,"""All Saints""","""Love Is Love""",2
5,"""KC & The Sunshine Band""","""Who Do Ya Love""",1
…,…,…,…
46,"""ABC""","""Love Conquers All""",0
47,"""Destiny's Child""","""Love""",0
48,"""Clay Aiken""","""Joy To The World""",0



  Collection: LONELINESS


index,artist,title,play_count
u32,str,str,i64
1,"""Skip James""","""How Long 'Buck'""",0
2,"""Guy Forsyth""","""Brownsville""",0
3,"""David Bowie""","""Cat People (Putting Out Fire) …",0



  Collection: MONEY


index,artist,title,play_count
u32,str,str,i64
1,"""Black Eyed Peas""","""Let's Get It Started""",32
2,"""Ok Go""","""Get Over It""",29
3,"""50 Cent""","""I Get Money""",9
4,"""Dee Dee Sharp""","""Ride!""",5
5,"""En Vogue""","""My Lovin' (You're Never Gonna …",4
…,…,…,…
46,"""Liberty X""","""Dirty Cash""",0
47,"""Mase""","""Do You Wanna Get $? [feat. Puf…",0
48,"""Boyz II Men""","""I Will Get There""",0


### Approach 3 -- Classification (MultinomialNB / Logistic / SGD / RandomForest on lyrics)

In [6]:
classifiers = ["nb", "logistic", "sgd", "forest"]
clf_results = collection_classification_compare(rs, KEYWORDS, n=10, neg_ratio=1, classifiers=classifiers)
for name in classifiers:
    print(f"\n{'='*60}\nClassifier: {name.upper()}\n{'='*60}")
    for kw, df in clf_results[name].items():
        print(f"\n  Collection: {kw.upper()}")
        display(df)


Classifier: NB

  Collection: LOVE


index,artist,title,play_count
u32,str,str,i64
1,"""The Clientele""","""Rain""",0
2,"""The Corrs""","""Rain ( LP Version )""",4
3,"""OutKast""","""A Life In The of Benjamin Andr…",0
4,"""Weird Al Yankovic""","""Albuquerque""",4
5,"""Goodie MoB""","""Hold On (Explicit)""",0
…,…,…,…
46,"""Common""","""G.O.D. (Gaining One's Definiti…",2
47,"""Big Moe""","""The Letter""",0
48,"""Erasure""","""Reunion""",0



  Collection: WAR


index,artist,title,play_count
u32,str,str,i64
1,"""Guerilla Maab""","""Hid'in""",0
2,"""Antigama""","""War""",0
3,"""Deviates""","""My Life""",12
4,"""The Corrs""","""Rain ( LP Version )""",4
5,"""Lil' Zane""","""Ways Of The World""",0
…,…,…,…
46,"""Deep Puddle Dynamics""","""The Scarecrow Speaks""",0
47,"""Keith Murray""","""Child Of The Streets (Man Chil…",0
48,"""Eyedea & Abilities""","""E&A Day""",0



  Collection: HAPPINESS


...

index,artist,title,play_count
u32,str,str,i64
1,"""Utopia""","""Play This Game (LP Version)""",0
2,"""Bon Jovi""","""Till We Ain't Strangers Anymor…",0
3,"""Lambert Hendricks & Ross""","""Charleston Alley""",0
4,"""Richard Ashcroft""","""World Keeps Turning""",0
5,"""Morcheeba""","""What New York Couples Fight Ab…",0
…,…,…,…
46,"""Big Punisher""","""Beware""",0
47,"""Bow Wow""","""Big Dreams""",0
48,"""Twisted Sister""","""Burn In Hell""",0



  Collection: LONELINESS


index,artist,title,play_count
u32,str,str,i64
1,"""Schizoid""","""The Big Picture""",0
2,"""Playa Fly""","""Ghetto Eyes""",0
3,"""Infamous Mobb""","""We Strive""",0
4,"""Shaman""","""Reason""",0
5,"""Matisyahu""","""WP""",0
…,…,…,…
46,"""Dr. Elmo""","""Grandma Got Run Over by a Rein…",0
47,"""Senser""","""States Of Mind""",0
48,"""A Tribe Called Quest""","""Butter""",0



  Collection: MONEY


index,artist,title,play_count
u32,str,str,i64
1,"""Lil Wyte""","""In The Streets""",0
2,"""Kate Ryan""","""The Rain""",0
3,"""Twista & The Speedknot Mobstaz""","""Warm Embrace (LP Version)""",0
4,"""FU-Schnickens""","""La Schmoove""",0
5,"""Consequence""","""Job Song""",0
…,…,…,…
46,"""EPMD""","""You're A Customer""",2
47,"""Lou Reed""","""Dirty Blvd.""",2
48,"""Lloyd Banks""","""Born Alone_ Die Alone""",3


### Comparison

Check overlap between the three approaches for each keyword.

In [7]:
import polars as pl

def summarize(df):
    if df is None or len(df) == 0:
        return 0, set()
    return len(df), set(zip(df["artist"], df["title"]))

methods = [("baseline", baseline_results), ("w2v", w2v_results)]
methods += [(f"clf_{name}", clf_results[name]) for name in classifiers]

rows = {}
for kw in KEYWORDS:
    rows[kw.upper()] = {label: summarize(res.get(kw))[0] for label, res in methods}

print("Result sizes (rows per keyword and method):")
display(pl.DataFrame([{"keyword": kw, **vals} for kw, vals in rows.items()]))

Result sizes (rows per keyword and method):


keyword,baseline,w2v,clf_nb,clf_logistic,clf_sgd,clf_forest
str,i64,i64,i64,i64,i64,i64
"""LOVE""",50,50,50,50,50,50
"""WAR""",50,5,50,50,50,50
"""HAPPINESS""",50,50,50,50,50,50
"""LONELINESS""",50,3,50,50,50,50
"""MONEY""",50,50,50,50,50,50
